In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [3]:
# Example dataset (replace with real data)
texts = [
    "I love programming",
    "I hate bugs",
    "Transformers are powerful",
    "Deep learning is awesome",
    "Natural language processing is fascinating"
]

# Example labels (for a classification task, e.g., sentiment analysis or entity classification)
labels = [1, 0, 1, 1, 1]  # 1: Positive, 0: Negative

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

# Pad sequences to ensure uniform input size
max_length = max([len(seq) for seq in sequences])  # Get the longest sequence length
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

# Vocabulary size (number of unique words)
vocab_size = len(tokenizer.word_index) + 1  # Adding 1 for padding token


In [5]:
def positional_encoding(seq_len, d_model):
    position = np.arange(seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return tf.constant(pe, dtype=tf.float32)


In [7]:
def build_transformer_model(input_shape, vocab_size, num_heads=4, ff_dim=128, num_blocks=2, dropout=0.1):
    inputs = layers.Input(shape=input_shape)

    # Positional Encoding
    pe = positional_encoding(input_shape[0], input_shape[1])
    x = layers.Embedding(vocab_size, input_shape[1])(inputs)
    x += pe  # Add positional encoding

    # Transformer blocks
    for _ in range(num_blocks):
        # Multi-Head Self Attention
        attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=ff_dim, dropout=dropout)(x, x)
        x = layers.LayerNormalization()(attention + x)  # Residual connection

        # Feed Forward Network
        ff = layers.Dense(ff_dim, activation='relu')(x)
        ff = layers.Dense(input_shape[1])(ff)
        x = layers.LayerNormalization()(ff + x)  # Residual connection

    # Output layer for classification
    x = layers.GlobalAveragePooling1D()(x)  # Pool across sequence length
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)  # For binary classification

    model = models.Model(inputs, outputs)
    return model


In [11]:
def build_transformer_model(input_length, vocab_size, embedding_dim=64, num_heads=4, ff_dim=128, num_blocks=2, dropout=0.1):
    inputs = layers.Input(shape=(input_length,))

    # Positional Encoding
    pe = positional_encoding(input_length, embedding_dim)
    x = layers.Embedding(vocab_size, embedding_dim)(inputs)
    x += pe  # Add positional encoding

    # Transformer blocks
    for _ in range(num_blocks):
        # Multi-Head Self Attention
        attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=ff_dim, dropout=dropout)(x, x)
        x = layers.LayerNormalization()(attention + x)  # Residual connection

        # Feed Forward Network
        ff = layers.Dense(ff_dim, activation='relu')(x)
        ff = layers.Dense(embedding_dim)(ff)
        x = layers.LayerNormalization()(ff + x)  # Residual connection

    # Output layer for classification
    x = layers.GlobalAveragePooling1D()(x)  # Pool across sequence length
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)  # For binary classification

    model = models.Model(inputs, outputs)
    return model


In [13]:
# Define input shape based on the padded sequences
input_length = padded_sequences.shape[1]

# Build the transformer model
model = build_transformer_model(input_length, vocab_size)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(padded_sequences, np.array(labels), epochs=10, batch_size=32)


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.2000 - loss: 0.7883
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8000 - loss: 0.5993
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8000 - loss: 0.5624
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8000 - loss: 0.5010
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8000 - loss: 0.5088
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8000 - loss: 0.5333
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8000 - loss: 0.5135
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8000 - loss: 0.5013
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8000 - loss: 0.4994
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.8000 - loss: 0.5086
